# A2-Lite — object-focused internal-feature teaching
Latest startup correction: **A2-LITE-FEATURE-RESTART-2026-10-15-r1**. Restricted loader: **A2-LITE-FEATURE-SAFE-LOAD-2026-10-14-r1**. Experiment remains **A2-LITE-512-FOREGROUND-FEATURE-KD-2026-10-14-r1**.

After yesterday's Colab runtime ended, select a GPU (A100 preferred) and run ALL sections 1–8. Colab may have deleted /content; section 3 now restores the exact source BEFORE checking the saved run, and restores local data from the verified archive or Drive candidates. Keep the base RUN_ID below. Section 2 compares the current runtime before expensive staging: matching hardware/software resumes the saved run; changed runtime gets a clearly named separate run only if neither main training arm has started. That fresh run repeats its diagnostics and never mixes yesterday's weights/cache. If main training already started on a different runtime, stop early and review instead of silently starting over. Old manifests and results are never rewritten. Do not change weights_only or restore an old CUDA stack.

Question: does internal-feature teaching help more than human labels alone? This is a NEW, matched pair, not another output-loss weight variant.

Teacher: original A2 epoch130, MobileNetV4 Medium + MonoDETR E3/D3, 1280×384. Student: the same A2-Lite E1/D1, 640×192, hidden width256, 50 queries. Both arms start from the original pruned, untrained initialization. No architecture, deployment graph or head changes.

**Control:** all human-label losses. **Feature KD:** exactly those losses + a box-focused cosine feature loss. Before training, calibrate its fixed scale on up to32 TRAIN frames so the median weighted KD/GT gradient-norm ratio in the visual encoder is5% at initialization. Then ramp0→the calibrated coefficient over128 updates; do not adapt it online or use validation to set it. No direct teacher classification, boxes, depth, dimensions or angle loss. No temperature applies. Both classes keep all human-label losses and receive equal KD class shares when eligible. Balanced frame groups do NOT mean balanced object counts.

Feature point: final visual-encoder memory (teacher final stage after3 layers; student final stage after1), split into four spatial levels. Sample the same normalized 3×3 grid inside each labeled 2D box, with align_corners=False, to align the two resolutions. Mean over points, levels and objects, then present classes. Teacher qualification uses correct class, confidence≥0.1, 2D IoU≥0.5 and positive geometry. There is no claim that hidden features are ground truth. Box interiors can include background/occlusion. No feature adapter, background/global KD or query-index assumption. This is a small foreground-feature hypothesis inspired by [FGD](https://openaccess.thecvf.com/content/CVPR2022/html/Yang_Focal_and_Global_Knowledge_Distillation_for_Detectors_CVPR_2022_paper.html), NOT a reproduction of its full method.

Fixed pilot: exact reviewed512 TRAIN frames, five epochs /2,560 updates per arm. Batch1, original GT weights, same seeded order/optimizer, frozen BatchNorm/early backbone, no augmentation/dropout/AMP. A fresh control on this runtime avoids relying on historical software equivalence. No old virtual environment, Torch/CUDA replacement or extension compilation. A100 preferred; no phone needed.

Evaluate the fixed epoch5 on all3,769 VAL images, without selecting checkpoints on validation. Benefit guardrail: all five AP metrics nonregressing, mean moderate3D AP gain≥0.1 point, nearby recall drop≤2 absolute percentage points per class. Nearby recall uses 2D IoU≥0.5, score≥0.001, Vehicle≤40m/Pedestrian≤30m; it is NOT a3D correctness measure. Relative improvement does not mean product or deployment qualification.

Run sections 1–8 after a cold reset. Passed smoke/tiny-fit checks are reused only for a matching saved runtime; a separate runtime run starts from the original initialization. Checkpoints save every 128 updates; at most 127 interrupted updates can repeat. No automatic longer run or phone deployment follows.

## 1. Drive, paths and all helpers
Only adjust paths if your Drive folders differ. Internal features require the original A2 **.pth**, not just ONNX. The teacher export is used only to verify exact signed teacher fixtures.


In [ ]:
# A2-LITE-512-FOREGROUND-FEATURE-KD-2026-10-14-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, re, shlex, shutil, subprocess, sys, zipfile
REVISION='A2-LITE-512-FOREGROUND-FEATURE-KD-2026-10-14-r1'
SESSION_REVISION='A2-LITE-FEATURE-RESTART-2026-10-15-r1'
RUN_ID='a2_lite_512_foreground_feature_kd_r1'
ALLOW_PRETRAINING_RESTART=True  # New named run on runtime drift ONLY before main training starts.
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID):raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
PILOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_recovery/a2_lite_640_recovery_kd_r1')
INITIALIZATION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1/a2_lite_initialization.pth')
TEACHER_CHECKPOINT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_gt/monodetr_a2_mnv4_vehicle_pedestrian_gt/checkpoint_epoch_130.pth')
TEACHER_EXPORT=None  # Default: use the exact teacher_export recorded in the original pilot.
COMPACT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_resolution/a2_lite_resolution_cpu_screen_r1')
SPLITS=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
SOURCE_REPO=Path('/content/MonoDETR_A2LITE_FEATURE_'+RUN_ID)
DATASET=Path('/content/kitti_a2_lite_accuracy_feature_'+RUN_ID)
CACHE=Path('/content/drive/MyDrive/mobile_adas3d_inputs/kitti_chen_a2_v1')
LOCAL_ARCHIVE=Path('/content/a2_lite_archive_local/kitti_chen_7481.zip')
CANDIDATES=[Path(p) for p in (
    '/content/kitti_a2_lite_accuracy_vehicle_a2_lite_512_vehicle_kd_r1',
    '/content/kitti_a2_lite_accuracy_angle001_a2_lite_512_angle001_kd_r1',
    '/content/kitti_a2_lite_accuracy_geometry_a2_lite_512_geometry_kd_r1',
    '/content/kitti_a2_lite_accuracy_recovery_a2_lite_640_recovery_kd_r1',
    '/content/kitti_a2_lite_accuracy_full_a2_lite_640_full_recovery_kd_r1',
    '/content/kitti_a2_lite_accuracy_cache_prepare_v1',
    '/content/kitti_m68','/content/kitti_m67','/content/kitti',
    '/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti')]
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_features')/RUN_ID
# Loader-only compatibility; preserves the original experiment signature and checkpoints.
LOADER_REVISION='A2-LITE-FEATURE-SAFE-LOAD-2026-10-14-r1'
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/run_a2_lite_features_safe.py'
SESSION_SCRIPT=MOBILE_REPO/'scripts/preflight_a2_lite_feature_session.py'
SESSION_PLAN=Path('/content')/(RUN_ID+'_session_plan.json')
OUTPUT_SELECTED=False
BOOTSTRAP_LOGS=Path('/content/a2_lite_feature_startup_logs')/RUN_ID
LOGS=BOOTSTRAP_LOGS;LOGS.mkdir(parents=True,exist_ok=True)
READY=False;TINY_PASSED=False;KD_READY=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command));path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',path,flush=True);tail=deque(maxlen=40)
    with path.open('w') as log:
        log.write(shlex.join(command)+'\n');log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        status=process.wait();log.write(f'\nEXIT CODE: {status}\n');log.flush()
    if status:raise RuntimeError(f'Exit {status}; full log={path}\n'+'\n'.join(tail))
    return path
def stage(name,*extra):
    if not READY and name!='bundle':raise RuntimeError('Run sections1 and2 first')
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--output',OUTPUT,'--device','cuda',*extra],name,MOBILE_REPO)
print(SESSION_REVISION,'\nLoader:',LOADER_REVISION,'\nExperiment:',REVISION,'\nFresh GT control versus GT + feature KD. No output KD.\nOutput:',OUTPUT)


## 2. Update, validate the GPU, and select a safe run
The startup logs first stay on local disk so an incompatible session does not write into yesterday's run. Pull before checking helpers. Keep Colab's current Torch/CUDA; ordinary preprocessing dependencies are pinned. The early plan checks the saved manifest without requiring yesterday's deleted local checkout.

If the runtime matches, keep the same run. If it changed and main training never started, the explicit policy in section 1 allows a deterministic `_restart_<hash>` output with separate source/data paths. Yesterday's diagnostics/cache stay intact; this fresh run repeats its own diagnostics and both matched arms. If either main arm already started, runtime drift stops here before costly staging. No frozen receipt or manifest is re-signed.


In [ ]:
READY=False
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use clean main.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only'],'update',MOBILE_REPO)
for helper in (SCRIPT,SESSION_SCRIPT):
    if not helper.is_file():raise FileNotFoundError(f'{helper}: open the latest published restart notebook from main')
for path in (PILOT,INITIALIZATION,TEACHER_CHECKPOINT,COMPACT,SPLITS/'train.txt',SPLITS/'val.txt'):
    if not path.exists():raise FileNotFoundError(f'{path}: correct section1 paths')
if PILOT.is_dir():PREVIOUS=json.loads((PILOT/'recovery_manifest.json').read_text())
else:
    with zipfile.ZipFile(PILOT) as archive:
        names=[n for n in archive.namelist() if n=='recovery_manifest.json' or n.endswith('/recovery_manifest.json')]
        if len(names)!=1:raise RuntimeError('Expected one original512 pilot manifest')
        PREVIOUS=json.loads(archive.read(names[0]))
if TEACHER_EXPORT is None:TEACHER_EXPORT=Path(PREVIOUS['teacher_export'])
if not TEACHER_EXPORT.exists():raise FileNotFoundError(f'{TEACHER_EXPORT}: set TEACHER_EXPORT to the original M68 teacher export folder/ZIP')
saved=PREVIOUS['cpu_environment']
packages=[name+'=='+saved[name] for name in ('numpy','timm','onnxruntime','pillow','scipy','opencv-python-headless')]+['pyyaml','scikit-image']
run_logged([PYTHON,'-m','pip','install','-q',*packages],'ordinary_dependencies')
run_logged([PYTHON,'-c',"import torch;assert torch.cuda.is_available(),'Select a GPU runtime';torch.cuda.init();print(torch.__version__,torch.cuda.get_device_name(0),'portable training; no native extension')"],'gpu_probe')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_feature_checkpoint.py','-v'],'checkpoint_loader_tests',MOBILE_REPO)
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_feature_session.py','-v'],'session_restart_tests',MOBILE_REPO)
plan_command=[PYTHON,'-u',SESSION_SCRIPT,'plan','--pilot',PILOT,'--output',OUTPUT,
              '--run-id',RUN_ID,'--repo',SOURCE_REPO,'--dataset',DATASET,
              '--initialization',INITIALIZATION,'--teacher-checkpoint',TEACHER_CHECKPOINT,
              '--report',SESSION_PLAN,'--device','cuda']
if ALLOW_PRETRAINING_RESTART:plan_command += ['--allow-pretraining-restart']
run_logged(plan_command,'select_session_run',MOBILE_REPO)
PLAN=json.loads(SESSION_PLAN.read_text())
RUN_ID=PLAN['run_id'];OUTPUT=Path(PLAN['output']);SOURCE_REPO=Path(PLAN['repo']);DATASET=Path(PLAN['dataset'])
if Path(PLAN['requested_dataset'])!=DATASET:CANDIDATES.insert(0,Path(PLAN['requested_dataset']))
LOGS=OUTPUT/'colab_logs';LOGS.mkdir(parents=True,exist_ok=True)
for path in BOOTSTRAP_LOGS.glob('*.log'):
    target=LOGS/path.name
    if target.exists() and target.read_bytes()!=path.read_bytes():raise RuntimeError(f'Preserve conflicting startup log: {target}')
    if not target.exists():shutil.copyfile(path,target)
receipts=OUTPUT/'session_receipts';receipts.mkdir(parents=True,exist_ok=True)
receipt=receipts/f"session_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.json"
receipt.write_bytes(SESSION_PLAN.read_bytes())
OUTPUT_SELECTED=True
READY=True
print('Session action:',PLAN['action'],'\nSelected run:',RUN_ID,'\nOutput:',OUTPUT)


## 3. Restore the exact source, then stage inputs efficiently
A cold reset removes local source and data. Restore the pinned disposable MonoDETR checkout FIRST; the original frozen-run checks happen only after it exists. No tracked edits are reset or overwritten.

Use a completed verified input archive if available. Otherwise use four copy workers and reuse verified local candidates. Do not build an archive in this GPU run. A cold reset requires restoring local data again; it does not mean repeating completed main-training updates. All input hashes and original TRAIN/VAL splits are checked again.


In [ ]:
if not READY:raise RuntimeError('Run sections1 and2 first')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_features.py','-v'],'feature_tests',MOBILE_REPO)
run_logged([PYTHON,'-u',SESSION_SCRIPT,'restore-source','--plan',SESSION_PLAN,'--device','cuda'],'restore_portable_source',MOBILE_REPO)
extra=[]
for candidate in CANDIDATES:extra += ['--candidate',candidate]
if (CACHE/'kitti_input_cache.json').is_file():
    if not (CACHE/'kitti_chen_7481.zip').is_file():raise RuntimeError('Completed cache receipt exists but archive is missing; preserve and review')
    run_logged([PYTHON,'-u',MOBILE_REPO/'scripts/cache_a2_lite_inputs.py','restore',
                '--pilot',PILOT,'--cache',CACHE,'--dataset',DATASET,'--local-archive',LOCAL_ARCHIVE],'restore_input_archive',MOBILE_REPO)
else:
    run_logged([PYTHON,'-u',MOBILE_REPO/'scripts/stage_a2_lite_data.py',
                '--pilot',PILOT,'--dataset',DATASET,'--workers','4',*extra],'parallel_staging',MOBILE_REPO)
stage('prepare','--pilot',PILOT,'--initialization',INITIALIZATION,
      '--teacher-checkpoint',TEACHER_CHECKPOINT,'--teacher-export',TEACHER_EXPORT,
      '--compact',COMPACT,'--repo',SOURCE_REPO,'--dataset',DATASET,'--split-dir',SPLITS,*extra)
MANIFEST=json.loads((OUTPUT/'feature_manifest.json').read_text())
assert MANIFEST['recipe']['prediction_kd_components']==[] and MANIFEST['fresh_control_required']
assert len(MANIFEST['pilot_ids'])==512 and MANIFEST['recipe']['epochs']==5
print('Manifest:',MANIFEST['revision'],'— updates per arm:',512*5)
print('GT losses:',MANIFEST['recipe']['gt_weights'])
print('Feature alignment:',MANIFEST['recipe']['feature_alignment'])


## 4. Original student parity and label-only tiny fit
The unchanged diagnostic checks the original640 fixtures and learnability on16 TRAIN frames/128 updates. These disposable diagnostic weights NEVER initialize either arm. A failure means stop and return evidence, not lower the threshold.


In [ ]:
stage('smoke')
stage('tiny')
TINY=json.loads((OUTPUT/'tiny_fit.json').read_text());TINY_PASSED=bool(TINY['passed'])
print(json.dumps({k:TINY[k] for k in ('passed','gt_loss_ratio','geometry_error_ratio','geometry_ratio_per_class','evaluation_scope')},indent=2))
if not TINY_PASSED:print('STOP: return section8 evidence.')


## 5. Cache teacher features once, then check student gradients
Verify the full A2 model against all16 exact signed ONNX fixtures. Use GPU inference once on512 TRAIN frames to cache detached object features; never create features from validation labels. The teacher has no optimizer. A verified local mirror avoids small Drive reads in the training loop.

The zero-update gradient check covers up to32 fixed TRAIN frames and records feature/GT gradient norms and their direction in the student visual encoder. Both classes must have coverage. It scales the feature loss once to target a5% median weighted-KD/GT encoder gradient-norm ratio at initialization (safety limit: scale≤1000). The scale is frozen for the entire pilot and checked on resume. This is units calibration, not PCGrad or validation-driven tuning. It does not guarantee a5% AdamW update share, continued gradient balance or class preservation. If the safety limit fails, stop and return evidence.


In [ ]:
KD_READY=False
if TINY_PASSED:
    stage('cache')
    CACHE_REPORT=json.loads((OUTPUT/'feature_cache_complete.json').read_text())
    print(json.dumps({k:CACHE_REPORT[k] for k in ('images','encoded_gt_counts','qualified_teacher_matches','kd_pilot_authorized','teacher_weights_buffers_unchanged')},indent=2))
    if CACHE_REPORT['kd_pilot_authorized']:
        stage('feature-smoke')
        CHECK=json.loads((OUTPUT/'feature_kd_smoke.json').read_text());KD_READY=bool(CHECK['passed'])
        assert CHECK['optimizer_steps']==0 and CHECK['weights_buffers_unchanged']
        assert CHECK['prediction_kd_components']==[]
        print(json.dumps(CHECK,indent=2))
    else:print('STOP: feature coverage failed; return section8 evidence.')
else:print('Skipped: tiny learnability check failed.')


## 6. Fresh matched pair — five epochs each
Train control and feature KD from identical original pruned weights:2,560 updates per arm. The tiny-fit and previous recovery checkpoints are never used as initialization. No output KD. Identical GT losses, optimizer and seeded frame order; no validation-driven selection. On rerun, completed arms are verified/reused and incomplete arms resume atomically. Do not change GPU/software mid-run.


In [ ]:
if KD_READY:
    stage('train','--role','control')
    stage('train','--role','kd')
else:print('No paired training started; return the preflight evidence.')


## 7. Evaluate fixed epoch5 on all3,769 VAL frames
Compare this fresh feature arm against its fresh label-only control. Historical output-KD results are context only. A higher Vehicle score cannot hide Pedestrian regression. Review the predefined AP and nearby-recall guardrails before any further training or export. One seed is a screening result, not a universal verdict on feature KD.


In [ ]:
if KD_READY:
    stage('evaluate','--role','control')
    stage('evaluate','--role','kd')
    stage('compare')
    REPORT=json.loads((OUTPUT/'feature_comparison.json').read_text())
    for name,value in REPORT['results'].items():
        print(name,json.dumps({'metrics':value['metrics'],
                              'nearby_recall':{c:value['nearby']['classes'][c]['near_recall'] for c in ('Vehicle','Pedestrian')}},indent=2))
    print(json.dumps({k:REPORT[k] for k in ('versus_label_only','benefit_guardrail_passed','next_review',
                                          'comparison_limit','full_training_authorized','deployment_qualified')},indent=2))
else:print('Validation skipped: preflight did not qualify training.')


## 8. Download results — also after failure
Return **a2_lite_feature_results.zip** after a selected run. It includes reports, predictions, feature-file hashes, gradient diagnostics, parent/session evidence and combined logs. Images, packages, feature tensors and checkpoints stay on Drive.

If startup stopped before selecting a safe run, return **a2_lite_feature_startup_failure.zip** instead. It contains local startup logs and the last local plan, without modifying yesterday's output. No phone test or full-data run starts automatically.


In [ ]:
BUNDLE_ROOT=OUTPUT if OUTPUT_SELECTED else BOOTSTRAP_LOGS
BUNDLE=BUNDLE_ROOT/('a2_lite_feature_results.zip' if OUTPUT_SELECTED else 'a2_lite_feature_startup_failure.zip')
try:
    if not OUTPUT_SELECTED:raise RuntimeError('No safe run selected; package local startup evidence only')
    if not SCRIPT.is_file():raise RuntimeError('Helper missing; package readable logs directly')
    stage('bundle')
except Exception as error:
    print('Bundle helper unavailable:',error,'— packaging readable reports/logs directly.')
    with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(BUNDLE_ROOT.rglob('*')):
            if path.is_file() and 'assets' not in path.relative_to(BUNDLE_ROOT).parts and path.suffix in ('.json','.csv','.txt','.log'):
                archive.write(path,path.relative_to(BUNDLE_ROOT))
        if not OUTPUT_SELECTED and SESSION_PLAN.is_file():archive.write(SESSION_PLAN,'last_local_session_plan.json')
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
